# Rollout Collection at Scale: real SFT trajectories from the real gemma-4-31b-it-qat-w4a16-ct

Runs the validated agent scaffold (search_similar_code removed, pytest output-truncation fix, async graph warning, skip_summarization: false, max_output_tokens: 8192, max_time_minutes: 6 -- see docs/STRATEGY.md for how each was found and confirmed via runs 1-8 of the submission-validation notebook) against all 52 locally-verified-resolvable official-pool tasks, for multiple rounds at a higher sampling temperature (0.8, vs. the real submission's 0.2) to get diverse rejection-sampling attempts per task.

This uses a SEPARATE Kaggle dataset (`noobhunterjay/gemma4-rollout-agent`) from the real submission (`noobhunterjay/gemma4-agent-submission`) -- same validated prompts/tools/eval_config, but higher temperature, since this run is for collecting training data, not for the scored submission itself.

Uses the real `Evaluator.run()` method (concurrency=4, display_mode='quiet'), the same code path validated crash-free in run 8 of the submission-validation notebook. Each round writes to its own results directory; a trajectory is kept as an SFT candidate if ANY round resolves it.

## 1. Environment Configuration and Package Installation

In [ ]:
import glob
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')

## 2. Rollout Agent Scaffold and Task Selection

In [ ]:
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

candidates = [
    p.parent for p in Path('/kaggle/input').rglob('agent.yaml')
    if 'swe_scaffold_v1_rollout' in p.read_text(encoding='utf-8')
]
assert candidates, 'Could not find the rollout-agent dataset under /kaggle/input -- check dataset_sources in kernel-metadata.json'
OUR_SUBMISSION_SRC = candidates[0]
print(f'Found rollout-agent scaffold at: {OUR_SUBMISSION_SRC}')

AGENT_DIR = WORKING_DIR / 'rollout_agent'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(OUR_SUBMISSION_SRC, AGENT_DIR)

TASKS_PATH = DATA_DIR / 'tasks.jsonl'
all_tasks = load_tasks(TASKS_PATH)
by_id = {t.instance_id: t for t in all_tasks}

# All 52 official-pool tasks confirmed locally resolvable via the reference gold
# patch (submission_drafts/gold_patch_check/gold_patch_results.json in the project
# repo) -- no point spending agent budget on a task whose local/real environment
# can't even run the KNOWN-CORRECT patch.
RESOLVABLE_IDS = [
    'fastapi_14258', 'fastapi_14372', 'fastapi_14851', 'fastapi_14873', 'fastapi_14986',
    'fastapi_15280', 'fastapi_15588', 'fastapi_15589', 'fastapi_15661', 'fastapi_15800',
    'requests_7309', 'requests_7315', 'requests_7427',
    'rich_2725', 'rich_2943', 'rich_3006', 'rich_3043', 'rich_3052', 'rich_3061', 'rich_3063',
    'rich_3067', 'rich_3105', 'rich_3180', 'rich_3278', 'rich_3454', 'rich_3468', 'rich_3470',
    'rich_3471', 'rich_3480', 'rich_3506', 'rich_3518', 'rich_3521', 'rich_3535', 'rich_3675',
    'rich_3676', 'rich_3718', 'rich_3772', 'rich_3777', 'rich_3882', 'rich_3894', 'rich_3905',
    'rich_3930', 'rich_3934', 'rich_3935', 'rich_3938', 'rich_3942', 'rich_3944', 'rich_3953',
    'rich_4006', 'rich_4075', 'rich_4076', 'rich_4077',
]
TASK_POOL = [by_id[i] for i in RESOLVABLE_IDS if i in by_id]

print(f'Agent directory: {AGENT_DIR}')
print(f'Loaded {len(all_tasks)} total tasks; rollout pool has {len(TASK_POOL)} known-resolvable tasks')
by_repo_count = {}
for t in TASK_POOL:
    by_repo_count[t.repo] = by_repo_count.get(t.repo, 0) + 1
print('By repo:', by_repo_count)

## 3. Start vLLM Server (real `gemma-4-31b-it-qat-w4a16-ct`)

In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

litellm.drop_params = True

TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
INFERENCE_API_KEY = 'EMPTY'

declared_model = validate_single_declared_model(AGENT_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=True,
    max_loras=8,
    max_lora_rank=128,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

models = server_instance.create_model_registry(
    aliases=[declared_model, TARGET_MODEL_NAME],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

## 4. Rollout Collection: multiple rounds over the full 52-task pool

Each round is a full `Evaluator.run()` pass over all 52 tasks at temperature 0.8, writing to its own results directory. A task is kept as an SFT candidate if ANY round resolves it (rejection sampling).

In [ ]:
import asyncio
import concurrent.futures
import json
import time
import traceback
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator


def run_sync(coro_fn, **kwargs):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(coro_fn(**kwargs))).result()
    return asyncio.run(coro_fn(**kwargs))


eval_config_file = AGENT_DIR / 'eval_config.yaml'
budget_overrides = {}
if eval_config_file.exists():
    raw_eval_cfg = yaml.safe_load(eval_config_file.read_text(encoding='utf-8'))
    eval_section = raw_eval_cfg.get('evaluation', raw_eval_cfg)
    if 'timeout_seconds' in eval_section:
        budget_overrides['timeout_seconds'] = int(eval_section['timeout_seconds'])
    if 'max_tool_calls' in eval_section:
        budget_overrides['max_tool_calls'] = int(eval_section['max_tool_calls'])
    if 'max_time_minutes' in eval_section:
        budget_overrides['max_time_minutes'] = float(eval_section['max_time_minutes'])
    turns_raw = eval_section.get('max_turns', eval_section.get('max_llm_calls'))
    if turns_raw is not None:
        budget_overrides['max_turns'] = int(turns_raw)
print(f'Budget overrides from eval_config.yaml: {budget_overrides}')

ROLLOUTS_DIR = WORKING_DIR / 'rollouts'
ROLLOUTS_DIR.mkdir(parents=True, exist_ok=True)
NUM_ROUNDS = 2

limits, gen_constraints = build_submission_limits()
all_round_results = []

for round_idx in range(NUM_ROUNDS):
    round_dir = ROLLOUTS_DIR / f'round_{round_idx}'
    print(f'\n===== Round {round_idx + 1}/{NUM_ROUNDS} ({len(TASK_POOL)} tasks) =====')
    eval_config = EvalConfig(
        tasks_path=TASKS_PATH,
        snapshots_dir=DATA_DIR / 'snapshots',
        results_dir=round_dir,
        submission_dir=AGENT_DIR,
        models=models,
        sandbox='subprocess',
        task_ids=[t.instance_id for t in TASK_POOL],
        concurrency=4,
        display_mode='quiet',
        limits=limits,
        generation_constraints=gen_constraints,
        adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(
            compaction_interval=15,
            overlap_size=2,
            token_threshold=14336,
            event_retention_size=5,
        ),
        graph_dir=str(DATA_DIR / 'graphs'),
        embeddings_dir=str(DATA_DIR / 'embeddings'),
        wheels_dir=DATA_DIR / 'wheels',
        verbose=False,
        **budget_overrides,
    )
    evaluator = Evaluator(eval_config)
    start = time.perf_counter()
    try:
        eval_result = run_sync(evaluator.run)
    except Exception:
        print(f'Round {round_idx} RAISED after {time.perf_counter() - start:.1f}s:')
        traceback.print_exc()
        raise
    dur = time.perf_counter() - start
    print(f'Round {round_idx + 1} completed in {dur:.1f}s: {eval_result.resolved}/{eval_result.total} resolved ({eval_result.resolution_rate:.1%})')
    for repo, stats in sorted(eval_result.by_repo().items()):
        print(f'  {repo}: {stats}')
    all_round_results.append({'round': round_idx, 'results_dir': str(round_dir), 'duration_seconds': dur,
                                'resolved': eval_result.resolved, 'total': eval_result.total})

print('\n===== All rounds complete =====')
print(json.dumps(all_round_results, indent=2))

## 5. Consolidate SFT Candidates

A task is kept as an SFT candidate if ANY round resolved it. Copies the winning patch + trace for each into `sft_candidates/` for easy download, and writes a manifest.

In [ ]:
SFT_DIR = WORKING_DIR / 'sft_candidates'
SFT_DIR.mkdir(parents=True, exist_ok=True)

manifest = []
for round_idx in range(NUM_ROUNDS):
    round_dir = ROLLOUTS_DIR / f'round_{round_idx}'
    results_path = round_dir / 'task_results.jsonl'
    if not results_path.exists():
        continue
    with open(results_path, encoding='utf-8') as f:
        for line in f:
            rec = json.loads(line)
            if not rec.get('resolved'):
                continue
            instance_id = rec['instance_id']
            safe_id = instance_id.replace('/', '__')
            patch_src = round_dir / 'patches' / f'{safe_id}.patch'
            trace_src = round_dir / 'traces' / f'trace_{safe_id}.json'
            dest_name = f'{safe_id}__round{round_idx}'
            if patch_src.exists():
                shutil.copy(patch_src, SFT_DIR / f'{dest_name}.patch')
            if trace_src.exists():
                shutil.copy(trace_src, SFT_DIR / f'{dest_name}.trace.json')
            manifest.append({'instance_id': instance_id, 'repo': rec['repo'], 'round': round_idx,
                              'duration_seconds': rec['duration_seconds'], 'tool_calls': rec['tool_calls']})

(SFT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
unique_resolved = {m['instance_id'] for m in manifest}
print(f'{len(manifest)} resolved attempts across all rounds, covering {len(unique_resolved)}/{len(TASK_POOL)} unique tasks')
print(f'SFT candidates written to {SFT_DIR}')